In [1]:
import random
from cgra import *
from kernels import *

In [2]:
SIZE = 24
kernel_name = "benchmarks/compigra_kernel/blas/qmmul/3x3/"
version = "_3_IJK" + str(SIZE)

In [3]:
# Global variables
CGRA_N_ROWS = 3
CGRA_N_COLS = 3
# Adress
first_addr = 20000

In [4]:
def printAsMatrix(array, rows, cols):
    for i in range(rows):
        print(array[i * cols:(i + 1) * cols])

In [5]:
# Data
def configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, bias_data, scale_data):
    # Clear memory values
    kernel_clear_memory(kernel_name, version=version)
    # Config values
    # ----------------------            
    first_addr_A = first_addr
    first_addr_B = first_addr_A + rowsA*colsA*4
    first_addr_bias = first_addr_B + colsA*colsB*4
    first_addr_scale = first_addr_bias + rowsA*4
    first_addr_C = first_addr_scale + colsB*4

    config_vals = [[] for i in range(CGRA_N_COLS)]


    #void qmmul( int A[NI][NK], int B[NK][NJ], int bias[NI], int scale[NJ], int C[NI][NJ]) 
    # 0: first_addr_A
    # 1: first_addr_B
    # 2: first_addr_bias
    # 3: first_addr_scale
    # 4: first_addr_C

    if version == "_3_IJK24":
        config_vals[0] = [first_addr_bias, first_addr_B] # 2,1
        config_vals[1] = [] #
        config_vals[2] = [first_addr_A, first_addr_B, first_addr_C, first_addr_scale] # 0, 1, 4, 3
    
    if version == "_3_IJK60":
        config_vals[0] = [first_addr_A]
        config_vals[1] = [first_addr_B]
        config_vals[2] = [first_addr_C]
    
    addr_config_loads = [0 for i in range(CGRA_N_COLS)]
    for i in range(CGRA_N_COLS):
        kernel_add_memory_region(kernel_name, addr_config_loads[i], config_vals[i], version=version)
        if i < CGRA_N_COLS -1:
            addr_config_loads[i+1] = addr_config_loads[i] + len(config_vals[i])*4
    # Load data
    kernel_add_memory_region(kernel_name, first_addr_A, A_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_B, B_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_bias, bias_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_scale, scale_data, version=version)
    kernel_add_memory_region(kernel_name, first_addr_C, C_data, version=version)
    # Config data address for direct loads
    return addr_config_loads

In [6]:
def runKernel(load_addrs, max_it=1000, pr=["ROUT","INST"], printVal=1):
    # Run kernel
    run(kernel_name, pr=pr, load_addrs=load_addrs, version=version, limit=max_it, printVal=printVal)

In [7]:
def getResult(first_addr_C, end_addr_C, rowsA, colsB):
    result = [0 for _ in range(rowsA*colsB)]
    with open( kernel_name + "/memory_out"+version+".csv", 'r') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for row in csv_reader:
            try:
                if(int(row[0]) >= first_addr_C) and (int(row[0]) < end_addr_C):
                    result[int((int(row[0]) - first_addr_C)/4)] = int(row[1])
            except ValueError:
                print("Error: Values in memory_out CSV file are not integers.")
    return result

In [8]:
def qmmul_cpu(A_data, B_data, rowsA, colsA, colsB, bias_data, scale_data):
    expected_res = [0 for _ in range(rowsA*colsB)]
    for rA in range(rowsA):
        for cB in range(colsB):
            sum = 0
            for cA in range(colsA):
                a = A_data[rA*colsA + cA] + bias_data[rA]
                b = B_data[cA*colsB + cB] * scale_data[cB]
                sum += a*b
            expected_res[rA*colsB + cB] = sum 
    return expected_res

In [9]:
# Test dimensions (4xXx4)
rowsA = SIZE
colsA = SIZE
colsB = SIZE

A_data = [random.randint(-100, 100) for _ in range(rowsA * colsA)]
B_data = [random.randint(-100, 100) for _ in range(colsA * colsB)]
bias_data = [random.randint(-100, 100) for _ in range(rowsA)]
scale_data = [random.randint(-100, 100) for _ in range(colsB)]
C_data = [random.randint(-100, 100) for _ in range(rowsA * colsB)]


A_data_cpy = A_data.copy()
B_data_cpy = B_data.copy()
bias_data_cpy = bias_data.copy()
scale_data_cpy = scale_data.copy()
C_data_cpy = C_data.copy()

load_addrs = configMemory(A_data, B_data, C_data, rowsA, colsA, colsB, bias_data, scale_data)

In [10]:
runKernel(load_addrs, max_it=2000000, printVal=0)
#estimatedConfigCycles(kernel_name, version)

Instr =  0 ( 0 )
[  15,   15, 20000]    [SADD R0 15 ZERO, SADD ROUT 15 ZERO, LWD R0 4]    
[24608,   16, 22304]    [LWD R0 4, SADD ROUT 16 ZERO, LWD R1 4]    
[  24,   16,   15]    [SADD R0 ZERO 24, SADD ROUT 16 ZERO, SADD R1 15 ZERO]    
Aprox cycles this pc: 4
-------
Instr =  1 ( 1 )
[61440, 65536, 61440]    [SLT R0 R0 12, SLT R0 RCB 12, SLT ROUT RCL 12]    
[  15,   15, 24800]    [SADD R2 15 ZERO, SADD ROUT 15 ZERO, LWD R0 4]    
[22304,   15, 65536]    [LWD R1 4, SADD ROUT 15 ZERO, SLT R0 RCL 12]    
Aprox cycles this pc: 3
-------
Instr =  2 ( 2 )
[  15,   15,   24]    [SADD R1 15 ZERO, SADD R2 15 ZERO, SADD R1 ZERO 24]    
[61440,   15,   15]    [SLT R1 RCR 12, SADD R0 15 ZERO, SADD ROUT 15 ZERO]    
[61440,   15, 65028]    [SLT ROUT RCR 12, SADD R0 15 ZERO, SADD ROUT 3588 RCB]    
Aprox cycles this pc: 1
-------
Instr =  3 ( 3 )
[65024, 65880, 24704]    [SADD ROUT 3584 RCT, SADD R0 344 R0, LWD R2 4]    
[61440, 61440,   15]    [SLT ROUT R2 12, SLT ROUT RCR 12, SWI R1 RCB]    
[

In [12]:
# Get result from CGRA
first_addr_C = first_addr + rowsA*colsA*4 + colsA*colsB*4 + rowsA*4 + colsB*4
result = getResult(first_addr_C, first_addr_C + rowsA*colsB*4, rowsA, colsB)

# Get cpu output
expected_res = qmmul_cpu(A_data_cpy, B_data_cpy, rowsA, colsA, colsB, bias_data_cpy, scale_data_cpy)

# Check result correctness
errors = 0
for i in range(len(expected_res)):
    if expected_res[i] != result[i]:
        errors += 1
if errors > 0:
    print("Err: " + str(errors))
    print("Expected: ")
    printAsMatrix(expected_res, rowsA, colsB)
    print("CGRA: ")
    printAsMatrix(result, rowsA, colsB)
else:
    print("OK")



OK
